In [1]:
# 네이버 영화 리뷰 데이터셋(NSMC)을 이용한 감성 분류 모델
import os
import re
import urllib.request
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. NSMC 데이터 다운로드 (github에서 바로 불러오기)
base_url = "https://raw.githubusercontent.com/e9t/nsmc/master/"
for fname in ["ratings_train.txt", "ratings_test.txt"]:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(base_url + fname, fname)

# 2. 데이터 로드 및 정제
def load_and_clean(path):
    df = pd.read_csv(path, sep="\t").dropna().drop_duplicates()
    # 한글과 공백만 남기고 정제
    df["document"] = df["document"].apply(lambda x: re.sub(r"[^가-힣\s]", "", str(x)).strip())
    return df

train_df = load_and_clean("ratings_train.txt")
test_df = load_and_clean("ratings_test.txt")

# 3. 벡터화 (TF-IDF 상위 10000단어 사용)
vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(train_df["document"])
X_test = vectorizer.transform(test_df["document"])

# 4. 로지스틱 회귀 모델 학습
y_train = train_df["label"]
y_test = test_df["label"]
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# 5. 평가
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"\n✅ 정확도: {acc:.4f}\n")

# 6. 사용자 문장 분류 함수
def predict_sentiment(sentence):
    text = re.sub(r"[^가-힣\s]", "", sentence).strip()
    vec = vectorizer.transform([text])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0][pred]
    label = "긍정" if pred == 1 else "부정"
    print(f"\n▶ 예시 문장: \"{sentence}\"\n→ 예측: {label} (확률: {prob:.2f})")

# 7. 테스트 문장 실행
predict_sentiment("완전 재미있었어요!")
predict_sentiment("그저 그랬어요.")


✅ 정확도: 0.7784


▶ 예시 문장: "완전 재미있었어요!"
→ 예측: 긍정 (확률: 0.96)

▶ 예시 문장: "그저 그랬어요."
→ 예측: 부정 (확률: 0.70)
